In [1]:
import numpy as np
import pandas as pd
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.metrics import precision_score, recall_score, f1_score, fbeta_score, accuracy_score, ConfusionMatrixDisplay, make_scorer, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import joblib


In [ ]:
models = {
    "count + none": make_pipeline(
        CountVectorizer(),
        LogisticRegression(max_iter=1000),
    ),
    "count + none + ngrams": make_pipeline(
        CountVectorizer(ngram_range=(1, 2)),
        LogisticRegression(max_iter=1000),
    ),
    "count + balanced": make_pipeline(
        CountVectorizer(),
        LogisticRegression(max_iter=1000, class_weight="balanced"),
    ),
    "tfidf + none": make_pipeline(
        TfidfVectorizer(),
        LogisticRegression(max_iter=1000),
    ),
    "tfidf + balanced": make_pipeline(
        TfidfVectorizer(),
        LogisticRegression(max_iter=1000, class_weight="balanced"),
    ),
    "tfidf + balanced + ngrams": make_pipeline(
        TfidfVectorizer(ngram_range=(1, 2)),
        LogisticRegression(max_iter=1000, class_weight="balanced"),
    ),
}

In [ ]:
f2_scorer = make_scorer(fbeta_score, beta=2)

cv_results = []

for name, model in models.items():
    scores = cross_validate(
        model, X_td, y_td, cv=cv,
        scoring={"precision": "precision",
                 "recall": "recall",
                 "f2": f2_scorer}
    )
    cv_results.append({
        "model": name,
        "precision": scores["test_precision"].mean(),
        "recall": scores["test_recall"].mean(),
        "f2": scores["test_f2"].mean(),
        "f2_std": scores["test_f2"].std(),
    })
cv_results_df = pd.DataFrame(cv_results).sort_values("f2", ascending=False)
cv_best_name = cv_results_df.iloc[0]["model"]
print("Best model (CV):", cv_best_name)

cv_results_df

In [ ]:
best_pipe = models[cv_best_name]

param_grid = {
    "tfidfvectorizer__ngram_range": [(1, 1), (1, 2)],
    "tfidfvectorizer__min_df": [1, 2],
    "tfidfvectorizer__sublinear_tf": [True, False],


    "logisticregression__C": [0.1, 1.0, 10.0],
    "logisticregression__class_weight": [None, "balanced"],
}

grid_search = GridSearchCV(
    estimator=best_pipe,
    param_grid=param_grid,
    cv=cv,
    scoring=f2_scorer,
    n_jobs=1,
    verbose=1,
    refit=True
)
grid_search.fit(X_td, y_td)

In [ ]:
print(f"Best accuracy score:{grid_search.best_score_:.4f}")
print("Best parameters configuration:")
for param, value in grid_search.best_params_.items():
    print(f"  {param}: {value}")

In [ ]:
grid_search_model = grid_search.best_estimator_

td_probs = cross_val_predict(
    grid_search_model, X_td, y_td, cv=cv, method="predict_proba"
)[:, 1]

threshold_results = []

for threshold in [0.35, 0.40, 0.42, 0.44, 0.46, 0.48, 0.50, 0.55]:
    y_pred = (td_probs >= threshold).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(y_td, y_pred),
        "recall": recall_score(y_td, y_pred),
        "f2": fbeta_score(y_td, y_pred, beta=2),
    })

threshold_df = pd.DataFrame(threshold_results)
threshold_df

In [ ]:
# FINAL_THRESHOLD = CHECK TABLE ABOVE
final_model = grid_search.best_estimator_

In [ ]:
joblib.dump(final_model, "../models/tfidf_balanced_ngrams.joblib")

In [ ]:
def plot_confusion(y_true, y_pred, title):
    """Confusion matrix with plain-language labels, counts and row percentages."""
    # Counts and row percentages (each row adds up to 100%)
    cm = confusion_matrix(y_true, y_pred)
    cm_pct = confusion_matrix(y_true, y_pred, normalize="true")

    # Unpack the four cells: correct no, false alarm, missed, correct yes
    tn, fp, fn, tp = cm.ravel()
    tn_pct, fp_pct, fn_pct, tp_pct = cm_pct.ravel()

    # Text inside each cell, in the same layout as the grid
    labels = [
        [f"Correct: no refund intent\n{tn} ({tn_pct:.0%})", f"False alarm\n{fp} ({fp_pct:.0%})"],
        [f"Missed refund intent\n{fn} ({fn_pct:.0%})", f"Correct: refund intent\n{tp} ({tp_pct:.0%})"],
    ]

    plt.figure(figsize=(7, 5))
    sns.heatmap(
        cm_pct,
        annot=labels,
        fmt="",
        cmap="Blues",
        vmin=0,
        vmax=1,
        cbar=False,
        xticklabels=["Predicted: no refund", "Predicted: refund"],
        yticklabels=["Actually: no refund", "Actually: refund"],
    )
    plt.title(title)
    plt.show()

In [ ]:
test_probs = final_model.predict_proba(X_test)[:, 1]
y_pred_test = (test_probs >= FINAL_THRESHOLD).astype(int)

print("precision:", precision_score(y_test, y_pred_test))
print("recall:", recall_score(y_test, y_pred_test))
print("f2:", fbeta_score(y_test, y_pred_test, beta=2))

plot_confusion(
    y_test,
    y_pred_test,
    f"Final model: {cv_best_name}, threshold {FINAL_THRESHOLD} (test set)",
)